# Comprendre le scraper du projet AO-BTP Copilot

Ce notebook explique **pas à pas** comment fonctionne `src/scraper.py`, le module qui
récupère les **avis d'appel d'offres (AO) publics togolais** sur
**marches-publics-togo.com** et ne garde que ceux du domaine **Travaux (BTP)**.

L'objectif : **comprendre le code**, pas le recopier. Chaque brique est donc découpée en
petits morceaux minimalistes.

## Ce qu'on va faire

1. **Récupérer** le HTML d'une page web
2. **Lire** ce HTML pour en extraire les données utiles (référence, titre, date...)
3. **Classer** chaque AO : c'est du BTP / Travaux ou pas ?
4. **Stocker** le résultat dans une base SQLite

> 🔒 **Aucun réseau nécessaire** : ce notebook utilise une *fixture* locale (un morceau
> de HTML sauvegardé avec le projet), donc tout fonctionne même sans internet.

## Le scraping en 30 secondes

Le *scraping* consiste à :

1. **Télécharger** une page web → c'est une simple chaîne de texte HTML
2. **Parcourir** cette chaîne avec un outil type **BeautifulSoup** (le « scanner »)
3. **Extraire** les morceaux qui nous intéressent (par exemple les cellules d'un tableau)

Deux outils suffisent (ce sont les seuls du projet) :

- `requests` → télécharger le HTML
- `beautifulsoup4` + `lxml` → analyser le HTML

## Étape 0 — Les imports

On importe d'abord les outils, puis on définit **l'adresse du site**.

In [ ]:
import re
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

# L'adresse de base du site qu'on va interroger
BASE_URL = "https://www.marches-publics-togo.com"

# Le chemin de la page qui liste les appels d'offres
CONSULTATIONS_PATH = "/consultations"


### Pourquoi un « User-Agent » ?

Quand on se connecte à un site, on lui dit qui on est via l'en-tête HTTP `User-Agent`.
Par politesse et par transparence, le scraper **s'annonce honnêtement** au lieu de se
faire passer pour un navigateur.

In [ ]:
HEADERS = {
    "User-Agent": "AO-BTP-Copilot/0.1 (usage non commercial)"
}

HEADERS

## Étape 1 — Récupérer le HTML

Le cœur de la récupération : une petite fonction qui télécharge une page et retourne son
HTML.

> 💡 Le `timeout` évite d'attendre *indéfiniment* si le site ne répond pas.

In [ ]:
REQUEST_TIMEOUT = 15

def fetch_html(url: str) -> str:
    """Télécharge une page web et retourne son HTML."""
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()   # lève une erreur si la réponse est mauvaise
    return response.text

print(fetch_html)

> 🔒 Ici, pour tester **sans internet**, on lit la fixture locale au lieu de télécharger
> la vraie page. C'est exactement le même format de HTML.

In [ ]:
# Le fichier HTML de test fourni avec le projet.
# On le cherche depuis le dossier du notebook OU depuis la racine du projet,
# pour que le notebook fonctionne quel que soit le dossier de lancement.
CANDIDATS = [
    Path("..") / "data" / "fixtures" / "consultations_list.html",   # lancé depuis notebooks/
    Path("data") / "fixtures" / "consultations_list.html",          # lancé depuis la racine
]

def charger_fixture() -> str:
    for chemin in CANDIDATS:
        if chemin.exists():
            return chemin.read_text(encoding="utf-8")
    raise FileNotFoundError(
        "Fixture introuvable. Lancez le notebook depuis notebooks/ ou la racine du projet."
    )

html = charger_fixture()
print(html[:600])   # on affiche seulement le début pour voir la structure

## Étape 2 — Analyser le HTML avec BeautifulSoup

Le HTML brut est une grosse chaîne de texte. On la transforme en **objet navigable** avec
`BeautifulSoup` :

- `soup.find("table")` → trouve **la première** balise `<table>`
- `soup.find_all("tr")` → trouve **toutes** les lignes du tableau
- `.get_text(strip=True)` → récupère le texte d'une balise sans espaces inutiles

### Notre page contient un tableau

Le site liste les AO dans un tableau HTML avec ces colonnes :
`Référence | Titre | Entité | Type | Statut | Date limite`.

In [ ]:
soup = BeautifulSoup(html, "lxml")

# On trouve le tableau et toutes ses lignes
table = soup.find("table")
rows = table.find_all("tr")

print(f"Le tableau contient {len(rows)} ligne(s), en-tête inclus.")
print("\nDétail de la 2e ligne (le 1er vrai AO) :")
print(rows[1])

### Extraire les informations d'une ligne

Chaque ligne (`<tr>`) est découpée en cellules (`<td>`). On peut les lire une par une.

In [ ]:
row = rows[1]                    # on prend une ligne d'exemple
cells = row.find_all("td")      # toutes les cellules de la ligne

print("Cellule 0 (référence) :", cells[0].get_text(strip=True))
print("Cellule 1 (titre)     :", cells[1].get_text(strip=True)[:60], "...")
print("Cellule 3 (type)      :", cells[3].get_text(strip=True))
print("Cellule 5 (date)      :", cells[5].get_text(strip=True))

### Le titre est aussi un lien 🔗

Dans le tableau, le titre est une balise `<a>` (un hyperlien). On en tire :

- le **texte** → le titre de l'AO
- l'attribut **`href`** → le lien vers la page de détail

`urljoin()` transforme un lien relatif (`/consultations/...`) en lien absolu
(`https://...`).

In [ ]:
titre_cell = cells[1]
lien = titre_cell.find("a")

titre = lien.get_text(strip=True)
href = lien["href"]                     # on lit l'attribut href
url_detail = urljoin(BASE_URL, href)    # lien relatif → lien complet

print("Titre       :", titre[:60], "...")
print("Lien relatif:", href[:50], "...")
print("Lien absolu :", url_detail[:70], "...")

## Étape 3 — Extraire toutes les lignes en une fonction

Maintenant qu'on sait lire une ligne, on généralise à **toutes** les lignes du tableau.
Le résultat est une liste d'objets `Consultation`.

> ℹ️ Un `dataclass` est un simple conteneur propre qui regroupe les informations d'un AO.

In [ ]:
@dataclass
class Consultation:
    reference: str
    titre: str
    entite: str | None
    type_marche: str | None
    statut: str | None
    date_limite: str | None
    url_detail: str
    scraped_at: str
    is_btp: bool = False
    btp_classification_source: str | None = None

In [ ]:
def parse_consultations_table(html: str) -> list[Consultation]:
    """Lit le tableau de la page /consultations et retourne les AO trouvés."""
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table")
    if table is None:
        return []   # pas de tableau → rien à extraire

    results = []
    now = datetime.now(timezone.utc).isoformat()

    for row in table.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 5:
            continue    # ligne d'en-tête ou incomplète → on ignore

        # Référence (colonne 0)
        reference = cells[0].get_text(strip=True)

        # Titre + lien (colonne 1)
        lien = cells[1].find("a")
        titre = lien.get_text(strip=True) if lien else cells[1].get_text(strip=True)
        href = lien["href"] if lien and lien.has_attr("href") else None
        url_detail = urljoin(BASE_URL, href) if href else ""

        # Les autres colonnes
        entite = cells[2].get_text(strip=True) or None
        type_marche = cells[3].get_text(strip=True) or None
        statut = cells[4].get_text(strip=True) or None
        date_limite = cells[5].get_text(strip=True) if len(cells) > 5 else None

        results.append(Consultation(
            reference=reference,
            titre=titre,
            entite=entite,
            type_marche=type_marche,
            statut=statut,
            date_limite=date_limite,
            url_detail=url_detail,
            scraped_at=now,
        ))

    return results


consultations = parse_consultations_table(html)
print(f"On a extrait {len(consultations)} AO(s) depuis le tableau :")
for c in consultations:
    print(f"  - [{c.reference}] {c.titre[:55]}... ({c.type_marche})")

## Étape 4 — Le plus intéressant : reconnaître les AO « Travaux » (BTP)

Le site indique un type de marché, mais **il arrive qu'il soit vide ou erroné** (ex. un
vrai AO travaux étiqueté « — »). On ne peut pas s'y fier à 100 %.

La solution du projet : un **classifieur par mots-clés**. On cherche dans le titre des
mots typiques du BTP : *travaux, construction, route, barrage, forage, assainissement...*

### La règle de décision (simple et lisible)

1. Si le site a renseigné le type → on s'y fie (`source = "site"`)
2. Sinon, si le titre contient un mot-clé BTP → c'est du Travaux (`source = "mots-clés"`)
3. Sinon → pas du BTP

In [ ]:
# Une liste de mots qui trahissent un projet de Travaux/BTP
BTP_KEYWORDS = [
    "travaux", "construction", "réhabilitation", "réfection",
    "bâtiment", "génie civil", "voirie", "assainissement",
    "forage", "route", "pont", "barrage", "irrigation",
]

# On transforme la liste en un "motif" (regex) qui matche l'un de ces mots
BTP_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in BTP_KEYWORDS) + r")\b",
    re.IGNORECASE,
)

print("Le motif regex :", BTP_PATTERN.pattern)

In [ ]:
def classify_btp(titre: str, type_marche_site: str | None) -> tuple[bool, str | None]:
    """Un AO est-il du BTP ? Retourne (oui/non, source)."""
    # 1. Si le site a renseigné le type, on lui fait confiance
    if type_marche_site and type_marche_site.strip() not in ("", "—"):
        is_travaux = type_marche_site.strip().lower() == "travaux"
        return is_travaux, ("site" if is_travaux else None)

    # 2. Sinon, on cherche les mots-clés BTP dans le titre
    if BTP_PATTERN.search(titre):
        return True, "mots-clés"

    # 3. Rien trouvé → pas du BTP
    return False, None


# Petits tests pour comprendre la règle :
exemples = [
    ("Réalisation de forages à énergie solaire", "Travaux"),     # le site dit Travaux
    ("Travaux de réfection de la voirie", None),                 # type vide, titre parlant
    ("Fourniture de mobiliers de bureau", "Fournitures"),        # pas du BTP
]
for titre, type_site in exemples:
    est_btp, source = classify_btp(titre, type_site)
    print(f"{est_btp!s:5} | source={source!s:10} | {titre} (type site: {type_site})")

### Brancher la classification sur le parsing

Dans la vraie fonction, chaque AO est classé au moment de l'extraction, grâce à son titre
et à son type :

In [ ]:
def parse_consultations_table_classified(html: str) -> list[Consultation]:
    """Même chose qu'avant, mais chaque AO est en plus classé BTP ou pas."""
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table")
    if table is None:
        return []

    results = []
    now = datetime.now(timezone.utc).isoformat()

    for row in table.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 5:
            continue

        titre = cells[1].get_text(strip=True)
        type_marche = cells[3].get_text(strip=True) or None

        is_btp, source = classify_btp(titre, type_marche)

        results.append(Consultation(
            reference=cells[0].get_text(strip=True),
            titre=titre,
            entite=cells[2].get_text(strip=True) or None,
            type_marche=type_marche,
            statut=cells[4].get_text(strip=True) or None,
            date_limite=cells[5].get_text(strip=True) if len(cells) > 5 else None,
            url_detail="",
            scraped_at=now,
            is_btp=is_btp,
            btp_classification_source=source,
        ))

    return results


resultats = parse_consultations_table_classified(html)
for c in resultats:
    tag = f"BTP ({c.btp_classification_source})" if c.is_btp else "autre"
    print(f"[{c.reference}] {tag:22} {c.titre[:45]}...")

## Étape 5 — Sauvegarder dans une base SQLite

Le but final : **stocker** les résultats pour pouvoir les réutiliser. SQLite est parfait
ici : un simple fichier, aucun serveur, très répandu.

On crée une table, puis on insère chaque AO. Avec `ON CONFLICT DO UPDATE`, si un AO a
déjà été vu, on met simplement ses informations à jour au lieu de le dupliquer.

In [ ]:
def save_to_sqlite(consultations: list[Consultation], db_path: str) -> None:
    """Écrit les AO dans un fichier SQLite."""
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS consultations (
            reference TEXT PRIMARY KEY,
            titre TEXT NOT NULL,
            entite TEXT,
            type_marche TEXT,
            statut TEXT,
            date_limite TEXT,
            url_detail TEXT NOT NULL,
            scraped_at TEXT NOT NULL,
            is_btp INTEGER NOT NULL DEFAULT 0,
            btp_classification_source TEXT
        )
    """)

    for c in consultations:
        ligne = asdict(c)
        ligne["is_btp"] = int(ligne["is_btp"])   # SQLite n'a pas de type booléen
        conn.execute("""
            INSERT INTO consultations
                (reference, titre, entite, type_marche, statut, date_limite,
                 url_detail, scraped_at, is_btp, btp_classification_source)
            VALUES (:reference, :titre, :entite, :type_marche, :statut, :date_limite,
                    :url_detail, :scraped_at, :is_btp, :btp_classification_source)
            ON CONFLICT(reference) DO UPDATE SET
                titre = excluded.titre,
                entite = excluded.entite,
                type_marche = excluded.type_marche,
                statut = excluded.statut,
                date_limite = excluded.date_limite,
                url_detail = excluded.url_detail,
                scraped_at = excluded.scraped_at,
                is_btp = excluded.is_btp,
                btp_classification_source = excluded.btp_classification_source
        """, ligne)

    conn.commit()
    conn.close()
    print(f"Enregistré : {len(consultations)} AO(s) dans {db_path}")


save_to_sqlite(resultats, "data/processed/demo_consultations.db")

### Vérifier que les données sont bien en base

On relit la base avec une simple requête SQL :

In [ ]:
conn = sqlite3.connect("data/processed/demo_consultations.db")
for reference, titre, is_btp, source in conn.execute(
    "SELECT reference, titre, is_btp, btp_classification_source FROM consultations"
):
    tag = "BTP" if is_btp else "---"
    print(f"{reference} | {tag:3} | {source!s:10} | {titre[:45]}...")
conn.close()

## Le scraper en une image

```
┌─────────────┐   ┌──────────────┐   ┌─────────────────┐   ┌────────────┐
│ 1. Récupérer │──▶│ 2. Lire      │──▶│ 3. Classer BTP  │──▶│ 4. Stocker │
│   le HTML    │   │  (Beautiful) │   │  (site/mots)    │   │  (SQLite)  │
└─────────────┘   └──────────────┘   └─────────────────┘   └────────────┘
   fetch_html()      parse_table()      classify_btp()      save_to_sqlite()
```

## Et le vrai module dans tout ça ?

Le notebook reprend l'essentiel de la logique. Le vrai `src/scraper.py` ajoute quelques
garde-fous en plus, à lire une fois les bases comprises :

- **`parse_consultations_cards()`** → un plan B si le site supprime le tableau un jour
- **`scrape_consultations()`** → assemble tout : récupère, parse, classifie, filtre
- **`main()`** → la ligne de commande (`python src/scraper.py --btp-only --out ...`)

Le fichier `tests/test_scraper.py` valide le parsing sur la même fixture que celle
utilisée ici. C'est ce qui garantit que le code reste correct quand on le modifie.